In [6]:
import torch
import triton
import triton.language as tl
from einops import rearrange
from typing import Literal, Optional
# test_conv2d.py
import torch
import triton
import triton.language as tl
import time
# from triton_conv2d import triton_conv2d
import argparse

In [7]:
# triton_conv2d.py
import torch
import triton
import triton.language as tl

@triton.jit
def triton_conv2d_kernel(
    # Pointers to matrices
    input_ptr, weight_ptr, output_ptr,
    # Matrix dimensions
    batch, out_channels, out_h, out_w,
    in_channels, in_h, in_w, kernel_h, kernel_w,
    stride_h, stride_w, padding_h, padding_w, dilation_h, dilation_w,
    groups,
    # Strides
    stride_input_b, stride_input_c, stride_input_h, stride_input_w,
    stride_weight_g, stride_weight_k, stride_weight_r, stride_weight_s,
    stride_output_b, stride_output_c, stride_output_h, stride_output_w,
    # Meta-parameters
    BLOCK_BATCH: tl.constexpr,
    BLOCK_OUT_C: tl.constexpr,
    BLOCK_IN_C: tl.constexpr,
    BLOCK_OUT_H: tl.constexpr,
    BLOCK_OUT_W: tl.constexpr,
    BLOCK_KERNEL_H: tl.constexpr,
    BLOCK_KERNEL_W: tl.constexpr,
):
    """
    Triton kernel for grouped 2D convolution forward pass.
    """
    # Program IDs
    pid_b = tl.program_id(0)  # batch
    pid_g = tl.program_id(1)  # group id
    pid_c = tl.program_id(2)  # output channel within group

    # Compute group-specific output channel range
    channels_per_group = out_channels // groups
    pid_out_c = pid_g * channels_per_group + pid_c

    # Load input base pointer
    input_base = input_ptr + pid_b * stride_input_b
    weight_base = weight_ptr + pid_g * stride_weight_g
    output_base = output_ptr + pid_b * stride_output_b + pid_out_c * stride_output_c

    # Block sizes
    BLOCK_BATCH = min(BLOCK_BATCH, batch)
    BLOCK_OUT_C = min(BLOCK_OUT_C, channels_per_group)
    BLOCK_IN_C = min(BLOCK_IN_C, in_channels // groups)
    BLOCK_KERNEL_H = min(BLOCK_KERNEL_H, kernel_h)
    BLOCK_KERNEL_W = min(BLOCK_KERNEL_W, kernel_w)

    # Iterate over output tiles
    offs_out_h = tl.arange(0, BLOCK_OUT_H)
    offs_out_w = tl.arange(0, BLOCK_OUT_W)

    # Output spatial indices
    out_h_mask = offs_out_h < out_h
    out_w_mask = offs_out_w < out_w

    # Input spatial indices
    # Calculate input position: (oh * stride - pad + kh * dilation)
    offs_input_h = offs_out_h[:, None] * stride_h - padding_h + tl.arange(0, BLOCK_KERNEL_H)[None, :] * dilation_h
    offs_input_w = offs_out_w[None, :] * stride_w - padding_w + tl.arange(0, BLOCK_KERNEL_W)[:, None] * dilation_w

    # Bounds check for input indices
    input_h_mask = (offs_input_h >= 0) & (offs_input_h < in_h)
    input_w_mask = (offs_input_w >= 0) & (offs_input_w < in_w)
    input_mask_hw = input_h_mask & input_w_mask  # [BLOCK_OUT_H, BLOCK_KERNEL_H, BLOCK_KERNEL_W]

    # Input pointer: [B, C, H, W]
    input_ptrs = input_base + \
        (offs_input_h * stride_input_h)[None, :, :, None] + \
        (offs_input_w * stride_input_w)[None, None, :, :]  # [B, IN_C, KH, KW]

    # Weight pointer: [G, K, R, S] where K = out_c_per_group
    offs_k = pid_c  # output channel in group
    offs_r = tl.arange(0, BLOCK_KERNEL_H)
    offs_s = tl.arange(0, BLOCK_KERNEL_W)
    weight_ptrs = weight_base + \
        offs_k * stride_weight_k + \
        offs_r[:, None] * stride_weight_r + \
        offs_s[None, :] * stride_weight_s  # [KH, KW]

    # Initialize accumulator
    acc = tl.zeros((BLOCK_OUT_H, BLOCK_OUT_W), dtype=tl.float32)

    # Iterate over input channels in group
    in_c_per_group = in_channels // groups
    for ic in range(0, in_c_per_group, BLOCK_IN_C):
        current_in_c_block = min(in_c_per_group - ic, BLOCK_IN_C)
        offs_in_c = ic + tl.arange(0, BLOCK_IN_C)

        # Input pointer for current input channel block
        mask_c = offs_in_c < in_c_per_group
        input_ptrs_c = input_ptrs + offs_in_c[None, :, None, None] * stride_input_c  # [B, IN_C_BLOCK, KH, KW]
        input_mask = mask_c[:, None, None] & input_mask_hw[None, ...]  # [IN_C_BLOCK, H, KH, KW]

        # Load input block
        input_tile = tl.load(input_ptrs_c, mask=input_mask, other=0.0)  # [IN_C_BLOCK, H, KH, KW]

        # Load weight block
        weight_mask = (offs_in_c < in_c_per_group)[:, None, None]  # [IN_C_BLOCK, KH, KW]
        weight_ptrs_c = weight_ptrs + offs_in_c[:, None, None] * stride_weight_k  # [IN_C_BLOCK, KH, KW]
        weight_tile = tl.load(weight_ptrs_c, mask=weight_mask, other=0.0)  # [IN_C_BLOCK, KH, KW]

        # Contract over input channels and kernel space
        # input_tile: [IN_C_BLOCK, H, KH, KW]
        # weight_tile: [IN_C_BLOCK, KH, KW]
        # We contract over IN_C_BLOCK, KH, KW
        contracted = tl.sum(input_tile * weight_tile[None, ...], axis=[0, 2, 3])  # [H, W]

        # Accumulate
        acc += contracted

    # Store output
    output_ptrs = output_base + \
        offs_out_h[:, None] * stride_output_h + \
        offs_out_w[None, :] * stride_output_w
    output_mask = out_h_mask[:, None] & out_w_mask[None, :]
    tl.store(output_ptrs, acc.to(input_ptr.dtype), mask=output_mask)

In [8]:
# triton_conv2d.py (continued)

def triton_conv2d(input, weight, bias=None, stride=1, padding=0, dilation=1, groups=1):
    """
    Triton-based grouped 2D convolution.
    """
    if isinstance(stride, int):
        stride = (stride, stride)
    if isinstance(padding, int):
        padding = (padding, padding)
    if isinstance(dilation, int):
        dilation = (dilation, dilation)

    stride_h, stride_w = stride
    padding_h, padding_w = padding
    dilation_h, dilation_w = dilation

    batch, in_channels, in_h, in_w = input.shape
    out_channels, _, kernel_h, kernel_w = weight.shape

    assert in_channels % groups == 0
    assert out_channels % groups == 0

    out_h = (in_h + 2 * padding_h - dilation_h * (kernel_h - 1) - 1) // stride_h + 1
    out_w = (in_w + 2 * padding_w - dilation_w * (kernel_w - 1) - 1) // stride_w + 1

    output = torch.empty((batch, out_channels, out_h, out_w), device=input.device, dtype=input.dtype)

    # Strides
    stride_input_b = input.stride(0)
    stride_input_c = input.stride(1)
    stride_input_h = input.stride(2)
    stride_input_w = input.stride(3)

    stride_weight_g = weight.stride(0)
    stride_weight_k = weight.stride(1)
    stride_weight_r = weight.stride(2)
    stride_weight_s = weight.stride(3)

    stride_output_b = output.stride(0)
    stride_output_c = output.stride(1)
    stride_output_h = output.stride(2)
    stride_output_w = output.stride(3)

    # Launch kernel
    def grid(meta):
        return (
            triton.cdiv(batch, meta['BLOCK_BATCH']),
            groups,
            triton.cdiv(out_channels // groups, meta['BLOCK_OUT_C'])
        )

    triton_conv2d_kernel[grid](
        input, weight, output,
        batch, out_channels, out_h, out_w,
        in_channels, in_h, in_w, kernel_h, kernel_w,
        stride_h, stride_w, padding_h, padding_w, dilation_h, dilation_w,
        groups,
        stride_input_b, stride_input_c, stride_input_h, stride_input_w,
        stride_weight_g, stride_weight_k, stride_weight_r, stride_weight_s,
        stride_output_b, stride_output_c, stride_output_h, stride_output_w,
        # Meta
        BLOCK_BATCH=4,
        BLOCK_OUT_C=16,
        BLOCK_IN_C=16,
        BLOCK_OUT_H=32,
        BLOCK_OUT_W=32,
        BLOCK_KERNEL_H=kernel_h,
        BLOCK_KERNEL_W=kernel_w,
        num_stages=3,
        num_warps=4,
    )

    if bias is not None:
        output += bias.view(1, -1, 1, 1)

    return output

In [9]:
def benchmark_torch_vs_triton(
    batch, in_channels, out_channels, in_h, in_w,
    kernel_h, kernel_w, stride, padding, dilation, groups,
    dtype=torch.float16, device='cuda'
):
    # Generate data
    x = torch.randn(batch, in_channels, in_h, in_w, dtype=dtype, device=device)
    w = torch.randn(out_channels, in_channels // groups, kernel_h, kernel_w, dtype=dtype, device=device)
    b = torch.randn(out_channels, dtype=dtype, device=device) if True else None

    # Triton forward
    try:
        z_triton = triton_conv2d(x, w, b, stride=stride, padding=padding, dilation=dilation, groups=groups)
        torch.cuda.synchronize()
        triton_start = time.time()
        for _ in range(50):
            z_triton = triton_conv2d(x, w, b, stride=stride, padding=padding, dilation=dilation, groups=groups)
        torch.cuda.synchronize()
        triton_time = (time.time() - triton_start) / 50
    except Exception as e:
        print(f"Triton failed: {e}")
        triton_time = float('inf')
        z_triton = None

    # PyTorch forward
    z_torch = torch.nn.functional.conv2d(x, w, b, stride=stride, padding=padding, dilation=dilation, groups=groups)
    torch.cuda.synchronize()
    torch_start = time.time()
    for _ in range(50):
        z_torch = torch.nn.functional.conv2d(x, w, b, stride=stride, padding=padding, dilation=dilation, groups=groups)
    torch.cuda.synchronize()
    torch_time = (time.time() - torch_start) / 50

    # Correctness
    if z_triton is not None:
        max_diff = torch.max(torch.abs(z_triton - z_torch)).item()
        mean_diff = torch.mean(torch.abs(z_triton - z_torch)).item()
    else:
        max_diff = float('inf')
        mean_diff = float('inf')

    # Speedup
    speedup = torch_time / triton_time if triton_time < torch_time else 0.0

    print(f"Config: B={batch}, C_in={in_channels}, C_out={out_channels}, HxW={in_h}x{in_w}, "
          f"K={kernel_h}x{kernel_w}, S={stride}, P={padding}, D={dilation}, G={groups} | "
          f"Max Diff: {max_diff:.2e}, Mean Diff: {mean_diff:.2e} | "
          f"Torch: {torch_time*1000:.2f}ms, Triton: {triton_time*1000:.2f}ms, "
          f"Speedup: {speedup:.2f}x")

    return max_diff, torch_time, triton_time



In [ ]:
# if __name__ == "__main__":
configs = [
    # Small kernel, common
    (4, 64, 64, 56, 56, 3, 3, 1, 1, 1, 1),
    (4, 64, 128, 56, 56, 3, 3, 2, 1, 1, 1),
    # Large kernel
    (2, 256, 256, 28, 28, 5, 5, 1, 2, 1, 1),
    # Deep stem
    (8, 3, 64, 224, 224, 7, 7, 2, 3, 1, 1),
    # Grouped conv (e.g., MobileNet)
    (4, 128, 128, 28, 28, 3, 3, 1, 1, 1, 8),
    (2, 256, 256, 14, 14, 3, 3, 1, 1, 1, 16),
    # Dilated conv
    (4, 64, 64, 56, 56, 3, 3, 1, 1, 2, 1),
    # Large batch
    (16, 64, 128, 32, 32, 3, 3, 1, 1, 1, 1),
    # Asymmetric kernel
    (4, 64, 64, 32, 32, 1, 5, 1, 0, 1, 1),
    (4, 64, 64, 32, 32, 5, 1, 1, 2, 1, 1),
]

print(f"{'='*120}")
print(f"{'Benchmark: Triton Conv2d vs PyTorch'}")
print(f"{'='*120}")

for config in configs:
    benchmark_torch_vs_triton(*config, dtype=torch.float16)

Benchmark: Triton Conv2d vs PyTorch
Triton failed: at 56:64:

    # Iterate over output tiles
    offs_out_h = tl.arange(0, BLOCK_OUT_H)
    offs_out_w = tl.arange(0, BLOCK_OUT_W)

    # Output spatial indices
    out_h_mask = offs_out_h < out_h
    out_w_mask = offs_out_w < out_w

    # Input spatial indices
    # Calculate input position: (oh * stride - pad + kh * dilation)
    offs_input_h = offs_out_h[:, None] * stride_h - padding_h + tl.arange(0, BLOCK_KERNEL_H)[None, :] * dilation_h
                                                                ^
Config: B=4, C_in=64, C_out=64, HxW=56x56, K=3x3, S=1, P=1, D=1, G=1 | Max Diff: inf, Mean Diff: inf | Torch: 0.05ms, Triton: infms, Speedup: 0.00x
Triton failed: at 56:64:

    # Iterate over output tiles
    offs_out_h = tl.arange(0, BLOCK_OUT_H)
    offs_out_w = tl.arange(0, BLOCK_OUT_W)

    # Output spatial indices
    out_h_mask = offs_out_h < out_h
    out_w_mask = offs_out_w < out_w

    # Input spatial indices
    # Calculate i

: 

In [13]:
import torch
import triton
import triton.language as tl
from einops import rearrange
from typing import Literal, Optional


@triton.autotune(
    configs=[
        triton.Config({"BLOCK_M": 128, "BLOCK_N": 256}, num_stages=3, num_warps=8),
        triton.Config({"BLOCK_M": 128, "BLOCK_N": 128}, num_stages=3, num_warps=8),
        triton.Config({"BLOCK_M": 128, "BLOCK_N": 64}, num_stages=3, num_warps=8),
        triton.Config({"BLOCK_M": 128, "BLOCK_N": 32}, num_stages=3, num_warps=8),
        triton.Config({"BLOCK_M": 64, "BLOCK_N": 128}, num_stages=3, num_warps=8),
        triton.Config({"BLOCK_M": 64, "BLOCK_N": 64}, num_stages=3, num_warps=8),
        triton.Config({"BLOCK_M": 64, "BLOCK_N": 32}, num_stages=3, num_warps=8),
        triton.Config({"BLOCK_M": 32, "BLOCK_N": 256}, num_stages=3, num_warps=8),
        triton.Config({"BLOCK_M": 32, "BLOCK_N": 128}, num_stages=3, num_warps=8),
        triton.Config({"BLOCK_M": 32, "BLOCK_N": 64}, num_stages=3, num_warps=8),
        triton.Config({"BLOCK_M": 32, "BLOCK_N": 32}, num_stages=3, num_warps=8),
    ],
    key=["seqlen", "dim", "batch"],
)
@triton.jit()
def _causal_conv1d_fwd_kernel(
    # Pointers to matrices
    x_ptr,  # (batch, dim, seqlen)
    w_ptr,  # (dim, width)
    bias_ptr,
    initial_states_ptr,
    o_ptr,  # (batch, dim, seqlen)
    # Matrix dimensions
    batch,
    dim,
    seqlen,
    # Strides
    stride_x_seq,  # stride to get to next sequence,
    stride_x_dim,  # stride to get to next feature-value,
    stride_x_token,  # stride to get to next token (same feature-index, same sequence-index)
    stride_weight_dim,  # stride to get to next dim-axis value
    stride_weight_width,  # stride to get to next width-axis value
    stride_istate_seq,
    stride_istate_dim,
    stride_istate_token,
    stride_o_seq,
    stride_o_dim,
    stride_o_token,
    # Meta-parameters
    HAS_BIAS: tl.constexpr,
    KERNEL_WIDTH: tl.constexpr,  # maybe using this we don't need 'width'
    SILU_ACTIVATION: tl.constexpr,
    HAS_INITIAL_STATES: tl.constexpr,
    BLOCK_M: tl.constexpr,
    BLOCK_N: tl.constexpr,
):
    indices_0 = tl.program_id(0) * BLOCK_M + tl.arange(0, BLOCK_M)
    idx_seqs = indices_0 // seqlen
    idx_tokens = indices_0 % seqlen

    x_base = x_ptr + (idx_seqs * stride_x_seq)[:, None]  # the beginning features at all tokens at all sequences processed by this Triton program
    idx_feats = tl.program_id(1) * BLOCK_N + tl.arange(0, BLOCK_N)
    w_base = w_ptr + (idx_feats * stride_weight_dim)  # first kernel column, configured for weights to handle BLOCK_N features in range
    load_init_state = False
    if HAS_INITIAL_STATES:
        load_init_state = tl.min(idx_tokens) < KERNEL_WIDTH - 1
        initial_states_base = initial_states_ptr + (idx_seqs * stride_istate_seq)[:, None] + (idx_feats * stride_istate_dim)[None, :]

    # store output data at the corresponding tokens (BLOCK_M of them) and feature-indices (BLOCK_N of them) in these tokens
    if HAS_BIAS:
        bias = bias_ptr + idx_feats
        mask_bias = idx_feats < dim
        acc = tl.load(bias, mask=mask_bias, other=0.0).to(tl.float32)[None, :]  # [BLOCK_N]
        acc = tl.broadcast_to(acc, (BLOCK_M, BLOCK_N))
    else:
        acc = tl.zeros((BLOCK_M, BLOCK_N), dtype=tl.float32)
    PADDING_W = KERNEL_WIDTH - 1
    for j in range(KERNEL_WIDTH):
        idx_x_w = j - PADDING_W + idx_tokens  # the token index to multiply with kernel[:, 0], given kernel with width-columns, i.e. kernel[:, 0..(width-1)]
        x_ptrs = x_base + ((idx_x_w * stride_x_token)[:, None] + (idx_feats * stride_x_dim)[None, :])  # [BLOCK_M, BLOCK_N]
        mask_x = ((idx_seqs < batch)[:, None]  # sequence-index
                  & (idx_x_w >= 0)[:, None]  # token-index
                  & (idx_x_w < seqlen)[:, None]  # token-index
                  & (idx_feats < dim)[None, :]  # feature-index
                  )
        if HAS_INITIAL_STATES:
            if load_init_state:
                initial_states_ptrs = initial_states_base + ((idx_x_w + KERNEL_WIDTH - 1) * stride_istate_token)[:, None]  # [BLOCK_M, BLOCK_N]
                mask_w = (idx_seqs < batch)[:, None] & (idx_x_w < 0)[:, None] & (idx_feats < dim)[None, :]  # sequence-index  # token-index  # feature-index
                initial_states = tl.load(initial_states_ptrs, mask_w, 0.0)
            else:
                initial_states = tl.zeros((BLOCK_M, BLOCK_N), dtype=x_ptr.dtype.element_ty)
            matrix_x = tl.load(x_ptrs, mask=mask_x, other=initial_states)
        else:
            matrix_x = tl.load(x_ptrs, mask=mask_x, other=0.0)

        w_ptrs = w_base[None, :] + \
            (j * stride_weight_width)  # [1, BLOCK_N] tensor
        mask_w = (idx_feats < dim)[None, :]
        matrix_w = tl.load(w_ptrs, mask_w, other=0.0)
        acc += matrix_x * matrix_w

    if SILU_ACTIVATION:
        acc = acc / (1 + tl.exp(-acc))
    mask = (
        (idx_seqs < batch)[:, None]  # sequence-index
        & (idx_tokens < seqlen)[:, None]  # token-index
        & (idx_feats < dim)[None, :]  # feature-index
    )
    o_ptrs = (
        o_ptr
        + (idx_seqs * stride_o_seq)[:, None]
        + (idx_tokens * stride_o_token)[:, None]
        + (idx_feats * stride_o_dim)[None, :]
    )

    tl.store(o_ptrs, acc, mask=mask)


def causal_conv1d_fwd(
    x: torch.Tensor,
    weight: torch.Tensor,
    bias: Optional[torch.Tensor] = None,
    seq_idx: Optional[torch.Tensor] = None,
    initial_states: Optional[torch.Tensor] = None,
    return_final_states: Optional[torch.Tensor] = False,
    final_states_out: Optional[torch.Tensor] = None,
    activation: Optional[Literal["silu", "swish"]] = None,
):
    batch, dim, seqlen = x.shape
    _, width = weight.shape
    assert (dim, width) == weight.shape
    assert x.stride(2) == 1 or x.stride(1) == 1
    # TODO: we may want to use weight such that weight.stride(dim)==1
    assert weight.stride(1) == 1
    # Tensor layout as NHWC is called channel last with 'C' is time-dimension
    is_channel_last = (x.stride(1) == 1) & (x.stride(2) > 1)
    stride_w_dim = weight.stride(0)
    stride_w_width = weight.stride(1)
    # effort to make data contiguous along dim-axis:
    weight = weight.transpose(0, 1).contiguous()
    stride_w_dim = weight.stride(1)
    stride_w_width = weight.stride(0)

    # assert initial_states is None  # only this for now
    assert return_final_states is False
    stride_istate_seq = 0
    stride_istate_dim = 0
    stride_istate_token = 0
    if initial_states is not None:
        assert (batch, dim, width - 1) == initial_states.shape
        stride_istate_seq = initial_states.stride(0)
        stride_istate_dim = initial_states.stride(1)
        stride_istate_token = initial_states.stride(2)
        assert stride_istate_dim == 1

    out = torch.empty_like(x)

    if not is_channel_last:
        assert 0, "Need to run in channel-last layout"
    else:

        def grid(META):
            return (
                triton.cdiv(batch * seqlen, META["BLOCK_M"]),
                triton.cdiv(dim, META["BLOCK_N"]),
            )

        with torch.cuda.device(x.device.index):
            _causal_conv1d_fwd_kernel[grid](
                # Pointers to matrices
                x,
                weight,
                bias,
                initial_states,
                out,
                # Matrix dimensions
                batch,
                dim,
                seqlen,
                # stride
                x.stride(0),
                x.stride(1),
                x.stride(2),
                stride_w_dim,
                stride_w_width,
                stride_istate_seq,
                stride_istate_dim,
                stride_istate_token,
                out.stride(0),
                out.stride(1),
                out.stride(2),
                # META
                HAS_BIAS=bias is not None,
                KERNEL_WIDTH=width,
                SILU_ACTIVATION=activation in ["silu", "swish"],
                HAS_INITIAL_STATES=initial_states is not None,
            )
    return out


class CausalConv1dFn(torch.autograd.Function):
    @staticmethod
    def forward(
        ctx,
        x,
        weight,
        bias=None,
        seq_idx=None,
        initial_states=None,
        return_final_states: bool = False,
        final_states_out=None,
        activation: Optional[Literal["silu", "swish"]] = None,
    ):
        # NOTE: in fact, 'beta=1' would turn swish into silu - and only silu form is used
        if x.stride(2) != 1 and x.stride(1) != 1:
            x = x.contiguous()
        bias = bias.contiguous() if bias is not None else None
        if seq_idx is not None:
            assert initial_states is None, "initial_states must be None if seq_idx is not None"
            assert not return_final_states, "If seq_idx is not None, we don't return final_states_out"
        seq_idx = seq_idx.contiguous() if seq_idx is not None else None
        if initial_states is not None and ((initial_states.stride(2) != 1) and (initial_states.stride(1) != 1)):
            initial_states = initial_states.contiguous()
        if return_final_states:
            assert (
                x.stride(1) == 1
            ), "Only channel-last layout support returning final_states_out"
            if final_states_out is not None:
                assert (
                    (final_states_out.stride(2) == 1) or (
                        final_states_out.stride(1) == 1)
                )
            else:
                batch, dim, seqlen = x.shape
                width = weight.shape[1]
                final_states_out = torch.empty(
                    batch, width - 1, dim, device=x.device, dtype=x.dtype).transpose(1, 2)
        else:
            final_states_out = None
        ctx.activation = activation
        out = causal_conv1d_fwd(
            x,
            weight,
            bias=bias,
            seq_idx=seq_idx,
            initial_states=initial_states,
            return_final_states=return_final_states,
            final_states_out=final_states_out,
            activation=ctx.activation,
        )
        ctx.save_for_backward(x, weight, bias, seq_idx, initial_states)
        ctx.return_final_states = return_final_states
        ctx.return_dinitial_states = initial_states is not None and initial_states.requires_grad
        return out if not return_final_states else (out, final_states_out)

    # @staticmethod
    # def backward(ctx, dout, *args):
    #     """dout = dL/dy
    #     RETURN: dL/dx, dL/dweight, dL/dbias, ...
    #     GIVEN THAT: def forward(ctx, x, weight, bias=None...)
    #     """
    #     x, weight, bias, seq_idx, initial_states = ctx.saved_tensors
    #     dfinal_states = args[0] if ctx.return_final_states else None
    #     if dout.stride(2) != 1 and dout.stride(1) != 1:
    #         dout = dout.contiguous()
    #     # The kernel supports passing in a pre-allocated dx (e.g., in case we want to fuse the
    #     # backward of conv1d with the backward of chunk).
    #     # Here we just pass in None and dx will be allocated in the C++ code.
    #     dx, dweight, dbias, dinitial_states = causal_conv1d_bwd(
    #         x,
    #         weight,
    #         bias,
    #         dout,
    #         seq_idx,
    #         initial_states,
    #         dfinal_states,
    #         None,
    #         ctx.return_dinitial_states,
    #         ctx.activation,
    #     )
    #     return (
    #         dx,
    #         dweight,
    #         dbias if bias is not None else None,
    #         None,
    #         dinitial_states if initial_states is not None else None,
    #         None,
    #         None,
    #         None,
    #     )


def causal_conv1d_fn(
    x,  # channel last, i.e. (batch, dim, seqlen)
    weight,  # (dim, w)
    bias=None,  # (dim, )scalar
    seq_idx=None,
    initial_states=None,
    return_final_states=False,
    final_states_out=None,
    activation: Optional[Literal["silu", "swish"]] = None,
):
    """causal_conv1d_fn.

    :param x: (batch, dim, seqlen) tensor
    :param weight: (dim, w) tensor
    :param bias: (dim,) tensor
    :param activation: ["silu", "swish"]
    :param seq_idx=None
    :param initial_states=None
    :param return_final_states=False
    :param final_states_out=None

    Return: (batch, dim, seqlen) tensor
    """
    if weight.dim() == 3:
        assert weight.shape[1] == 1
        weight = rearrange(weight, "d 1 w -> d w")
    return CausalConv1dFn.apply(
        x,
        weight,
        bias,
        seq_idx,
        initial_states,
        return_final_states,
        final_states_out,
        activation,
    )

In [ ]:
"""
test_causal_conv1d_vs_torch.py
"""
import torch
import torch.nn.functional as F
from torch import nn
import time
import numpy as np
# from causal_conv1d import causal_conv1d_fn   # 假设你把上面代码保存为 causal_conv1d.py

# ------------------------------------------------------------------
# 工具：把 PyTorch Conv1d 做成严格 causal
# ------------------------------------------------------------------
class TorchCausalConv1d(nn.Module):
    def __init__(self, dim, kernel_size, bias=True):
        super().__init__()
        self.pad = kernel_size - 1
        self.conv = nn.Conv1d(dim, dim, kernel_size, bias=bias, groups=dim)

    def forward(self, x):
        # x: (B, D, L)
        return self.conv(F.pad(x, (self.pad, 0)))[..., :x.shape[-1]]


# ------------------------------------------------------------------
# 误差统计
# ------------------------------------------------------------------
def relative_error(x, y, eps=1e-6):
    return (torch.abs(x - y) / (torch.abs(y).clamp_min(eps))).max().item()


# ------------------------------------------------------------------
# 单条测试
# ------------------------------------------------------------------
# def run_once(B, D, L, W, device="cuda", dtype=torch.float32, use_cache=False):
#     torch.manual_seed(123)
#     x = torch.randn(B, D, L, device=device, dtype=dtype, requires_grad=False)
#     w_torch = TorchCausalConv1d(D, W, bias=True).to(device, dtype)
#     w = w_torch.conv.weight.squeeze(1).detach()        # (D, W)
#     bias = w_torch.conv.bias.detach()                  # (D,)

#     # -------------- PyTorch 结果 --------------
#     y_torch = w_torch(x)
#     g = torch.randn_like(y_torch)
#     # y_torch.backward(g)
#     # dx_torch = x.grad.clone()
#     # dw_torch = w_torch.conv.weight.grad.clone()
#     # db_torch = w_torch.conv.bias.grad.clone()

#     x.grad = None
#     w_torch.zero_grad()

#     # -------------- Triton kernel --------------
#     x_triton = x.detach().clone().requires_grad_(False)
#     w_triton = w.detach().clone().requires_grad_(False)
#     bias_triton = bias.detach().clone().requires_grad_(False)

#     if use_cache:
#         # 前向时返回 final_states，反向时把 dfinal_states 传回
#         initial_states = torch.zeros(B, D, W-1, device=device, dtype=dtype)
#         y_triton, final_states = causal_conv1d_fn(
#             x_triton, w_triton, bias_triton,
#             initial_states=initial_states,
#             return_final_states=True)
#         y_triton.backward(g)
#         dx_triton = x_triton.grad.clone()
#         dw_triton = w_triton.grad.clone()
#         db_triton = bias_triton.grad.clone()

#         # 现在把 final_states 作为下一次的 initial_states
#         initial_states = final_states.detach()
#         # 第二次前向用 cache
#         y_triton2, final_states2 = causal_conv1d_fn(
#             x_triton, w_triton, bias_triton,
#             initial_states=initial_states,
#             return_final_states=True)
#     else:
#         y_triton = causal_conv1d_fn(
#             x_triton, w_triton, bias_triton)
#         # y_triton.backward(g)
#         # dx_triton = x_triton.grad.clone()
#         # dw_triton = w_triton.grad.clone()
#         # db_triton = bias_triton.grad.clone()

#     # -------------- 误差 --------------
#     fwd_err = relative_error(y_triton, y_torch)
#     # dx_err  = relative_error(dx_triton, dx_torch)
#     # dw_err  = relative_error(dw_triton, dw_torch)
#     # db_err  = relative_error(db_triton, db_torch)

#     return dict(fwd=fwd_err, 
#                 # dx=dx_err, 
#                 # dw=dw_err, 
#                 # db=db_err
#                 )




In [22]:
def run_once(B, D, L, W, device="cuda", dtype=torch.float32, use_cache=False):
    torch.manual_seed(123)
    # ----------- 构造数据 (channel-last) -----------
    x = torch.randn(B, L, D, device=device, dtype=dtype, requires_grad=False)   # (B, L, D)
    # 权重形状仍为 (D, W)，但 Conv1d 期望 (D, D, W)，这里用 groups=D 实现 depth-wise
    w_torch = nn.Conv1d(D, D, W, groups=D, bias=True).to(device, dtype)
    w = w_torch.weight.squeeze(1).detach()          # (D, W)
    bias = w_torch.bias.detach()                    # (D,)

    # PyTorch Conv1d 需要 channel-first，转一下
    x_cf = x.transpose(1, 2).contiguous()           # (B, D, L)

    # -------------- PyTorch 结果 --------------
    with torch.no_grad():
        # 把权重形状改成 (D, 1, W) 才能喂给 Conv1d
        w_torch.weight[:] = w.unsqueeze(1)
        w_torch.bias[:]   = bias
    y_torch = w_torch(x_cf)                         # (B, D, L)
    # g = torch.randn_like(y_torch)
    # y_torch.backward(g)
    # dx_torch = x_cf.grad.clone()
    # dw_torch = w_torch.weight.grad.clone()          # (D, 1, W)
    # db_torch = w_torch.bias.grad.clone()

    # -------------- Triton kernel --------------
    x_triton = x.detach().clone().requires_grad_(False)
    w_triton = w.detach().clone().requires_grad_(False)
    bias_triton = bias.detach().clone().requires_grad_(False)

    if use_cache:
        initial_states = torch.zeros(B, D, W-1, device=device, dtype=dtype)
        y_triton, final_states = causal_conv1d_fn(
            x_triton, w_triton, bias_triton,
            initial_states=initial_states,
            return_final_states=True)
    else:
        y_triton = causal_conv1d_fn(
            x_triton, w_triton, bias_triton)
    # y_triton.backward(g.transpose(1, 2))            # g 也要转回 channel-last
    # dx_triton = x_triton.grad.clone()
    # dw_triton = w_triton.grad.clone()
    # db_triton = bias_triton.grad.clone()

    # -------------- 误差 --------------
    fwd_err = relative_error(y_triton.transpose(1, 2), y_torch)
    # dx_err  = relative_error(dx_triton.transpose(1, 2), dx_torch)
    # dw_err  = relative_error(dw_triton, dw_torch.squeeze(1))
    # db_err  = relative_error(db_triton, db_torch)

    return dict(fwd=fwd_err, 
                # dx=dx_err, 
                # dw=dw_err, 
                # db=db_err
                )

In [23]:
# ------------------------------------------------------------------
# 多组参数测试
# ------------------------------------------------------------------
# if __name__ == "__main__":
device = "cuda"
dtype  = torch.float32
configs = [
    (1, 512, 1024, 4),
    (2, 768, 2048, 3),
    (4, 1024, 512, 5),
]

print(">>> without cache")
for B, D, L, W in configs:
    err = run_once(B, D, L, W, device, dtype, use_cache=False)
    print(f"B{B} D{D} L{L} W{W}  "
            f"fwd {err['fwd']:.3e}  dx {err['dx']:.3e}  dw {err['dw']:.3e}  db {err['db']:.3e}")

# print("\n>>> with cache")
# for B, D, L, W in configs:
#     err = run_once(B, D, L, W, device, dtype, use_cache=True)
#     print(f"B{B} D{D} L{L} W{W}  "
#             f"fwd {err['fwd']:.3e}  dx {err['dx']:.3e}  dw {err['dw']:.3e}  db {err['db']:.3e}")

>>> without cache


AssertionError: 

In [2]:
@triton.jit
def _causal_conv1d_group_fwd_kernel(
    x_ptr, w_ptr, bias_ptr, initial_states_ptr, o_ptr,
    # 维度
    batch, groups, out_c_per_group, seqlen,
    # strides
    stride_x_batch, stride_x_group, stride_x_c, stride_x_t,
    stride_w_group, stride_w_outc, stride_w_width,
    stride_istate_batch, stride_istate_group, stride_istate_c, stride_istate_t,
    stride_o_batch, stride_o_group, stride_o_c, stride_o_t,
    # meta
    HAS_BIAS: tl.constexpr,
    KERNEL_WIDTH: tl.constexpr,
    SILU_ACTIVATION: tl.constexpr,
    HAS_INITIAL_STATES: tl.constexpr,
    BLOCK_M: tl.constexpr,
    BLOCK_N: tl.constexpr,
):
    pid0 = tl.program_id(0)  # token
    pid1 = tl.program_id(1)  # group
    pid2 = tl.program_id(2)  # channel-within-group

    # 全局索引
    idx_tokens = pid0 * BLOCK_M + tl.arange(0, BLOCK_M)
    idx_batch  = idx_tokens // seqlen
    idx_t      = idx_tokens % seqlen

    idx_group   = pid1
    idx_local_c = pid2 * BLOCK_N + tl.arange(0, BLOCK_N)
    idx_global_c = idx_group * out_c_per_group + idx_local_c

    # 基地址
    x_base = x_ptr + idx_batch * stride_x_batch + idx_group * stride_x_group + idx_local_c * stride_x_c
    w_base = w_ptr + idx_group * stride_w_group + idx_local_c * stride_w_outc
    o_base = o_ptr + idx_batch * stride_o_batch + idx_group * stride_o_group + idx_local_c * stride_o_c

    if HAS_INITIAL_STATES:
        istate_base = initial_states_ptr + idx_batch * stride_istate_batch + idx_group * stride_istate_group + idx_local_c * stride_istate_c

    # 初始化累加器
    if HAS_BIAS:
        bias = tl.load(bias_ptr + idx_group * out_c_per_group + idx_local_c, mask=idx_local_c < out_c_per_group, other=0.0)
        acc = bias[None, :]
    else:
        acc = tl.zeros((BLOCK_M, BLOCK_N), dtype=tl.float32)

    # causal 卷积循环
    PADDING = KERNEL_WIDTH - 1
    for j in range(KERNEL_WIDTH):
        hist_idx = idx_t + j - PADDING
        # 计算地址
        x_ptrs = x_base + hist_idx[:, None] * stride_x_t
        w_ptrs = w_base + j * stride_w_width

        mask_x = (idx_batch < batch)[:, None] & (hist_idx >= 0)[:, None] & (hist_idx < seqlen)[:, None] & (idx_local_c < out_c_per_group)[None, :]
        matrix_w = tl.load(w_ptrs, mask=idx_local_c < out_c_per_group, other=0.0)

        if HAS_INITIAL_STATES:
            mask_istate = (idx_batch < batch)[:, None] & (hist_idx < 0)[:, None] & (idx_local_c < out_c_per_group)[None, :]
            istate_ptrs = istate_base + (hist_idx + PADDING)[:, None] * stride_istate_t
            init_val = tl.load(istate_ptrs, mask=mask_istate, other=0.0)
            matrix_x = tl.load(x_ptrs, mask=mask_x, other=init_val)
        else:
            matrix_x = tl.load(x_ptrs, mask=mask_x, other=0.0)

        acc += matrix_x * matrix_w

    if SILU_ACTIVATION:
        acc = acc * tl.sigmoid(acc)   # SiLU / Swish

    # 写回
    mask_o = (idx_batch < batch)[:, None] & (idx_t < seqlen)[:, None] & (idx_local_c < out_c_per_group)[None, :]
    o_ptrs = o_base + idx_t[:, None] * stride_o_t
    tl.store(o_ptrs, acc, mask=mask_o)

In [7]:
import torch
import triton
import triton.language as tl

@triton.jit
def _causal_conv1d_group_fwd_kernel(
    x_ptr, w_ptr, bias_ptr, initial_states_ptr, o_ptr,
    batch, groups, out_c_per_group, seqlen,
    stride_x_b, stride_x_g, stride_x_c, stride_x_l,
    stride_w_g, stride_w_c, stride_w_w,
    stride_is_b, stride_is_g, stride_is_c, stride_is_l,
    stride_o_b, stride_o_g, stride_o_c, stride_o_l,
    HAS_BIAS: tl.constexpr,
    KERNEL_WIDTH: tl.constexpr,
    SILU_ACTIVATION: tl.constexpr,
    HAS_INITIAL_STATES: tl.constexpr,
    BLOCK_M: tl.constexpr,
    BLOCK_N: tl.constexpr,
):
    pid0 = tl.program_id(0)   # token
    pid1 = tl.program_id(1)   # group
    pid2 = tl.program_id(2)   # channel-within-group

    idx_tok = pid0 * BLOCK_M + tl.arange(0, BLOCK_M)
    idx_b   = idx_tok // seqlen
    idx_l   = idx_tok %  seqlen

    idx_g   = pid1
    idx_c   = pid2 * BLOCK_N + tl.arange(0, BLOCK_N)

    # 掩码
    mask_tok = idx_tok < batch * seqlen
    mask_c   = idx_c < out_c_per_group

    # 累加器
    acc = tl.zeros((BLOCK_M, BLOCK_N), dtype=tl.float32)
    if HAS_BIAS:
        bias = tl.load(bias_ptr + idx_g * out_c_per_group + idx_c,
                       mask=mask_c, other=0.0)
        acc += bias[None, :]

    # causal 卷积
    pad = KERNEL_WIDTH - 1
    for k in range(KERNEL_WIDTH):
        l_in = idx_l + k - pad
        ptr_x = (x_ptr
                 + idx_b * stride_x_b
                 + idx_g * stride_x_g
                 + idx_c * stride_x_c
                 + l_in[:, None] * stride_x_l)

        mask_x = mask_tok[:, None] & (l_in >= 0)[:, None] & mask_c[None, :]
        if HAS_INITIAL_STATES:
            ptr_is = (initial_states_ptr
                      + idx_b * stride_is_b
                      + idx_g * stride_is_g
                      + idx_c * stride_is_c
                      + (l_in + pad)[:, None] * stride_is_l)
            val_is = tl.load(ptr_is, mask=mask_x, other=0.0)
            val_x  = tl.load(ptr_x,  mask=mask_x, other=val_is)
        else:
            val_x  = tl.load(ptr_x,  mask=mask_x, other=0.0)

        w_k = tl.load(w_ptr + idx_g * stride_w_g
                      + idx_c * stride_w_c
                      + k * stride_w_w,
                      mask=mask_c, other=0.0)
        acc += val_x * w_k[None, :]

    if SILU_ACTIVATION:
        acc = acc * tl.sigmoid(acc)

    # 写回
    ptr_o = (o_ptr
             + idx_b * stride_o_b
             + idx_g * stride_o_g
             + idx_c * stride_o_c
             + idx_l[:, None] * stride_o_l)
    mask_o = mask_tok[:, None] & mask_c[None, :]
    tl.store(ptr_o, acc, mask=mask_o)

In [5]:
def causal_conv1d_group_fwd(
    x: torch.Tensor,              # (B, C, L) 或 (B, groups, CpG, L)
    weight: torch.Tensor,         # (groups, CpG, width)
    bias: Optional[torch.Tensor] = None,
    initial_states: Optional[torch.Tensor] = None,
    activation: Optional[str] = None,
    groups: int = 1,
):
    # 统一处理成 4-D
    if x.dim() == 3:
        B, C, L = x.shape
        assert C % groups == 0
        x = x.view(B, groups, C // groups, L)
    elif x.dim() == 4:
        assert x.size(1) == groups
    else:
        raise ValueError("x must be 3-D or 4-D")
    # if x.dim() == 3:
    #     x = x.view(x.size(0), groups, x.size(1) // groups, x.size(2))
    batch, g, cpg, seqlen = x.shape
    width = weight.size(-1)

    out = torch.empty_like(x)
    grid = lambda META: (
        triton.cdiv(batch * seqlen, META["BLOCK_M"]),
        g,
        triton.cdiv(cpg, META["BLOCK_N"]),
    )
    _causal_conv1d_group_fwd_kernel[grid](
        x, weight, bias, initial_states, out,
        batch, g, cpg, seqlen,
        x.stride(0), x.stride(1), x.stride(2), x.stride(3),
        weight.stride(0), weight.stride(1), weight.stride(2),
        *(initial_states.stride(i) for i in range(4)) if initial_states is not None else (0, 0, 0, 0),
        out.stride(0), out.stride(1), out.stride(2), out.stride(3),
        HAS_BIAS=bias is not None,
        KERNEL_WIDTH=width,
        SILU_ACTIVATION=activation in ["silu", "swish"],
        HAS_INITIAL_STATES=initial_states is not None,
        BLOCK_M=128, BLOCK_N=64,  # 可在 autotune 里调
    )
    return out.view(batch, g * cpg, seqlen) if groups > 1 else out

In [8]:
def causal_conv1d_group_fwd(
    x: torch.Tensor,          # (B, C, L) 或 (B, groups, CpG, L)
    weight: torch.Tensor,     # (groups, CpG, width)
    bias: Optional[torch.Tensor] = None,
    initial_states: Optional[torch.Tensor] = None,
    activation: Optional[str] = None,
    groups: int = 1,
):
    if x.dim() == 3:
        B, C, L = x.shape
        assert C % groups == 0
        x = x.view(B, groups, C // groups, L).contiguous()
    elif x.dim() == 4:
        assert x.size(1) == groups
    else:
        raise ValueError("x must be 3-D or 4-D")

    B, G, CpG, L = x.shape
    width = weight.size(-1)
    out = torch.empty_like(x)

    grid = lambda META: (
        triton.cdiv(B * L, META["BLOCK_M"]),
        G,
        triton.cdiv(CpG, META["BLOCK_N"]),
    )

    _causal_conv1d_group_fwd_kernel[grid](
        x, weight, bias, initial_states, out,
        B, G, CpG, L,
        x.stride(0), x.stride(1), x.stride(2), x.stride(3),
        weight.stride(0), weight.stride(1), weight.stride(2),
        *(initial_states.stride() if initial_states is not None else (0, 0, 0, 0)),
        out.stride(0), out.stride(1), out.stride(2), out.stride(3),
        HAS_BIAS=bias is not None,
        KERNEL_WIDTH=width,
        SILU_ACTIVATION=activation in ["silu", "swish"],
        HAS_INITIAL_STATES=initial_states is not None,
        BLOCK_M=128, BLOCK_N=64,
    )
    return out.view(B, G * CpG, L)

In [6]:
B, C, L, W, G = 4, 512, 1024, 4, 8
x  = torch.randn(B, C, L, device='cuda', dtype=torch.float16)
w  = torch.randn(G, C // G, W, device='cuda', dtype=torch.float16)
b  = torch.randn(G, C // G, device='cuda', dtype=torch.float16)

y = causal_conv1d_group_fwd(x, w, b, groups=G, activation='silu')

CompilationError: at 32:13:

    # 全局索引
    idx_tokens = pid0 * BLOCK_M + tl.arange(0, BLOCK_M)
    idx_batch  = idx_tokens // seqlen
    idx_t      = idx_tokens % seqlen

    idx_group   = pid1
    idx_local_c = pid2 * BLOCK_N + tl.arange(0, BLOCK_N)
    idx_global_c = idx_group * out_c_per_group + idx_local_c

    # 基地址
    x_base = x_ptr + idx_batch * stride_x_batch + idx_group * stride_x_group + idx_local_c * stride_x_c
             ^
ValueError('Cannot make_shape_compatible: incompatible dimensions at index 0: 128 and 64')

In [9]:
B, C, L, G, W = 4, 512, 1024, 8, 4
x = torch.randn(B, C, L, device='cuda', dtype=torch.float16)
w = torch.randn(G, C//G, W, device='cuda', dtype=torch.float16)
b = torch.randn(G, C//G, device='cuda', dtype=torch.float16)

y = causal_conv1d_group_fwd(x, w, b, groups=G, activation='silu')
print(y.shape)   # torch.Size([4, 512, 1024])

CompilationError: at 41:17:
    # 累加器
    acc = tl.zeros((BLOCK_M, BLOCK_N), dtype=tl.float32)
    if HAS_BIAS:
        bias = tl.load(bias_ptr + idx_g * out_c_per_group + idx_c,
                       mask=mask_c, other=0.0)
        acc += bias[None, :]

    # causal 卷积
    pad = KERNEL_WIDTH - 1
    for k in range(KERNEL_WIDTH):
        l_in = idx_l + k - pad
        ptr_x = (x_ptr
                 ^
ValueError('Cannot make_shape_compatible: incompatible dimensions at index 0: 128 and 64')